# 作业2：多层感知机与正则化

丁平尖
2026年5月18日

## 2 多层感知机

### 2.1 理论计算题

#### 1. 非线性激活函数的重要性

假设一个具有单隐藏层的多层感知机，输入为 $x$，隐藏层没有激活函数（即线性激活），表达为 $h = W_1 x + b_1$，输出层为 $o = W_2 h + b_2$。

**证明过程：**

将隐藏层表达式代入输出层：

$o = W_2 h + b_2 = W_2 (W_1 x + b_1) + b_2$

展开后：

$o = W_2 W_1 x + W_2 b_1 + b_2$

因此，等价的单层神经网络为：

$o = W' x + b'$

其中：
- 权重矩阵：$W' = W_2 W_1$
- 偏置向量：$b' = W_2 b_1 + b_2$

**结论：** 没有非线性激活函数的多层感知机等价于单层神经网络，无法学习复杂的非线性映射关系。

#### 2. 激活函数性质分析

##### Sigmoid函数

**数学表达式：**

\[
\text{Sigmoid}(x) = \sigma(x) = \frac{1}{1 + e^{-x}}
\]

**导数推导：**

\[
\sigma'(x) = \frac{d}{dx} \frac{1}{1 + e^{-x}} = \frac{e^{-x}}{(1 + e^{-x})^2}
\]

注意到 $\sigma(x)(1 - \sigma(x)) = \frac{1}{1 + e^{-x}} \cdot \frac{e^{-x}}{1 + e^{-x}} = \frac{e^{-x}}{(1 + e^{-x})^2}$

因此：

\[
\sigma'(x) = \sigma(x)(1 - \sigma(x))
\]

##### tanh函数

**数学表达式：**

\[
\tanh(x) = \frac{e^x - e^{-x}}{e^x + e^{-x}}
\]

**导数推导：**

\[
\tanh'(x) = \frac{d}{dx} \frac{e^x - e^{-x}}{e^x + e^{-x}} = \frac{(e^x + e^{-x})^2 - (e^x - e^{-x})^2}{(e^x + e^{-x})^2}
\]

化简分子：$(e^x + e^{-x})^2 - (e^x - e^{-x})^2 = 4$

因此：

\[
\tanh'(x) = \frac{4}{(e^x + e^{-x})^2} = 1 - \tanh^2(x)
\]

### 2.2 编程题：从零实现单隐藏层MLP

In [ ]:
import numpy as np
import torch
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# 设置随机种子
np.random.seed(42)
torch.manual_seed(42)

In [ ]:
# 加载Fashion-MNIST数据集
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))  # 归一化到[-1, 1]
])

trainset = torchvision.datasets.FashionMNIST(
    root='./data', train=True, download=True, transform=transform
)
testset = torchvision.datasets.FashionMNIST(
    root='./data', train=False, download=True, transform=transform
)

# 数据转为numpy数组
X_train = trainset.data.numpy().reshape(-1, 28*28).astype(np.float64) / 255.0
y_train = trainset.targets.numpy()
X_test = testset.data.numpy().reshape(-1, 28*28).astype(np.float64) / 255.0
y_test = testset.targets.numpy()

print(f"训练集大小: {X_train.shape}")
print(f"测试集大小: {X_test.shape}")
print(f"类别数: {len(np.unique(y_train))}")

In [ ]:
# 手动初始化参数
input_size = 28 * 28
hidden_size = 256
output_size = 10

# 使用正态分布随机初始化
np.random.seed(42)
w1 = np.random.randn(input_size, hidden_size) * 0.01
b1 = np.zeros(hidden_size)
w2 = np.random.randn(hidden_size, output_size) * 0.01
b2 = np.zeros(output_size)

print(f"w1 shape: {w1.shape}")
print(f"b1 shape: {b1.shape}")
print(f"w2 shape: {w2.shape}")
print(f"b2 shape: {b2.shape}")

In [ ]:
# ReLU激活函数
def relu(x):
    return np.maximum(0, x)

# Softmax函数
def softmax(z):
    # 数值稳定性：减去最大值
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

# 交叉熵损失
def cross_entropy_loss(y_pred, y_true):
    n = y_true.shape[0]
    # 防止log(0)
    y_pred = np.clip(y_pred, 1e-10, 1 - 1e-10)
    # 取对应类别的概率
    log_probs = -np.log(y_pred[np.arange(n), y_true])
    return np.mean(log_probs)

# 前向传播
def forward(X, w1, b1, w2, b2):
    z1 = X @ w1 + b1
    h1 = relu(z1)
    z2 = h1 @ w2 + b2
    y_pred = softmax(z2)
    return z1, h1, z2, y_pred

# 计算准确率
def accuracy(y_pred, y_true):
    return np.mean(np.argmax(y_pred, axis=1) == y_true)

In [ ]:
# 训练循环
def train_mlp(X_train, y_train, w1, b1, w2, b2, epochs=10, batch_size=64, lr=0.1):
    n_samples = X_train.shape[0]
    loss_history = []
    acc_history = []
    
    for epoch in range(epochs):
        # 打乱数据
        indices = np.random.permutation(n_samples)
        X_shuffled = X_train[indices]
        y_shuffled = y_train[indices]

        epoch_loss = 0.0
        epoch_acc = 0.0
        n_batches = 0

        for i in range(0, n_samples, batch_size):
            X_batch = X_shuffled[i:i+batch_size]
            y_batch = y_shuffled[i:i+batch_size]

            # 前向传播
            z1, h1, z2, y_pred = forward(X_batch, w1, b1, w2, b2)

            # 计算损失
            loss = cross_entropy_loss(y_pred, y_batch)
            epoch_loss += loss
            epoch_acc += accuracy(y_pred, y_batch)
            n_batches += 1

            # 反向传播
            n = X_batch.shape[0]

            # 输出层梯度
            dz2 = y_pred.copy()
            dz2[np.arange(n), y_batch] -= 1
            dz2 /= n

            dw2 = h1.T @ dz2
            db2 = np.sum(dz2, axis=0)

            # 隐藏层梯度
            dh1 = dz2 @ w2.T
            dz1 = dh1 * (z1 > 0)  # ReLU导数

            dw1 = X_batch.T @ dz1
            db1 = np.sum(dz1, axis=0)

            # SGD更新
            w1 -= lr * dw1
            b1 -= lr * db1
            w2 -= lr * dw2
            b2 -= lr * db2

        avg_loss = epoch_loss / n_batches
        avg_acc = epoch_acc / n_batches
        loss_history.append(avg_loss)
        acc_history.append(avg_acc)

        print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}, Acc: {avg_acc:.4f}")

    return w1, b1, w2, b2, loss_history, acc_history

# 训练模型
w1_trained, b1_trained, w2_trained, b2_trained, loss_history, acc_history = train_mlp(
    X_train, y_train, w1, b1, w2, b2, epochs=20, batch_size=64, lr=0.5
)

In [ ]:
# 在测试集上评估
_, _, _, y_pred_test = forward(X_test, w1_trained, b1_trained, w2_trained, b2_trained)
test_acc = accuracy(y_pred_test, y_test)
test_loss = cross_entropy_loss(y_pred_test, y_test)

print(f"测试集准确率: {test_acc:.4f}")
print(f"测试集损失: {test_loss:.4f}")

# 绘制损失曲线
plt.figure(figsize=(10, 5))
plt.plot(loss_history, label='训练损失')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('MLP训练损失曲线')
plt.legend()
plt.show()

## 3 模型选择，权重衰减和丢弃法

### 3.1 理论计算题

#### 1. 过拟合与欠拟合

**训练误差（Training Error）**：模型在训练数据集上的误差，衡量模型对训练数据的拟合程度。

**泛化误差（Generalization Error）**：模型在未见过的测试数据集上的误差，衡量模型的推广能力。

**当训练误差极低但泛化误差很高时**：模型处于**过拟合**状态。这意味着模型过于复杂，学习到了训练数据中的噪声和细节，而不是数据的本质规律。

**缓解方法**：
- **降低模型复杂度**：减少隐藏层数量、减少隐藏单元数
- **正则化**：添加L1/L2正则化项
- **Dropout**：随机失活部分神经元
- **数据增强**：增加训练数据多样性
- **早停**：在验证集性能下降时停止训练

#### 2. K折交叉验证

**K折交叉验证算法步骤：**

1. **数据划分**：将训练数据集随机划分为K个大小大致相等的子集（称为"折"），记为 $D_1, D_2, \dots, D_K$。

2. **循环训练与验证**：对于每个 $i \in \{1, 2, \dots, K\}$：
   - 以 $D_i$ 作为验证集
   - 以 $\bigcup_{j \neq i} D_j$（即除 $D_i$ 外的所有其他折）作为训练集
   - 在训练集上训练模型
   - 在验证集 $D_i$ 上评估模型性能，记录指标（如准确率、损失等）

3. **结果汇总**：
   - 计算K次验证的性能指标的平均值，作为模型的最终性能估计
   - 可同时计算标准差，评估模型性能的稳定性

**优点**：
- 充分利用数据，每个样本都参与训练和验证
- 结果更可靠，降低随机性影响

**常用K值**：通常取K=5或K=10

### 3.2 编程题：添加L2正则化和Dropout

In [ ]:
# 带权重衰减的SGD
def train_mlp_with_weight_decay(X_train, y_train, w1, b1, w2, b2, epochs=10, batch_size=64, lr=0.1, weight_decay=0.0):
    n_samples = X_train.shape[0]
    loss_history = []
    acc_history = []
    
    for epoch in range(epochs):
        indices = np.random.permutation(n_samples)
        X_shuffled = X_train[indices]
        y_shuffled = y_train[indices]

        epoch_loss = 0.0
        epoch_acc = 0.0
        n_batches = 0

        for i in range(0, n_samples, batch_size):
            X_batch = X_shuffled[i:i+batch_size]
            y_batch = y_shuffled[i:i+batch_size]

            z1, h1, z2, y_pred = forward(X_batch, w1, b1, w2, b2)

            loss = cross_entropy_loss(y_pred, y_batch)
            epoch_loss += loss
            epoch_acc += accuracy(y_pred, y_batch)
            n_batches += 1

            n = X_batch.shape[0]
            dz2 = y_pred.copy()
            dz2[np.arange(n), y_batch] -= 1
            dz2 /= n

            dw2 = h1.T @ dz2
            db2 = np.sum(dz2, axis=0)

            dh1 = dz2 @ w2.T
            dz1 = dh1 * (z1 > 0)

            dw1 = X_batch.T @ dz1
            db1 = np.sum(dz1, axis=0)

            # 带权重衰减的SGD更新
            # 权重首先乘以 (1 - eta * lambda)
            w1 = (1 - lr * weight_decay) * w1 - lr * dw1
            b1 -= lr * db1
            w2 = (1 - lr * weight_decay) * w2 - lr * dw2
            b2 -= lr * db2

        avg_loss = epoch_loss / n_batches
        avg_acc = epoch_acc / n_batches
        loss_history.append(avg_loss)
        acc_history.append(avg_acc)

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}, Acc: {avg_acc:.4f}")

    return w1, b1, w2, b2, loss_history, acc_history

In [ ]:
# Dropout层实现
def dropout_layer(X, dropout, is_training=True):
    if not is_training or dropout == 0:
        return X
    # 创建随机掩码
    mask = np.random.rand(*X.shape) > dropout
    # 进行缩放
    return X * mask / (1 - dropout)

# 带Dropout的前向传播
def forward_with_dropout(X, w1, b1, w2, b2, dropout=0.5, is_training=True):
    z1 = X @ w1 + b1
    h1 = relu(z1)
    h1_dropout = dropout_layer(h1, dropout, is_training)
    z2 = h1_dropout @ w2 + b2
    y_pred = softmax(z2)
    return z1, h1, z2, y_pred

# 带Dropout的训练
def train_mlp_with_dropout(X_train, y_train, w1, b1, w2, b2, epochs=10, batch_size=64, lr=0.1, dropout=0.5):
    n_samples = X_train.shape[0]
    loss_history = []
    acc_history = []
    
    for epoch in range(epochs):
        indices = np.random.permutation(n_samples)
        X_shuffled = X_train[indices]
        y_shuffled = y_train[indices]

        epoch_loss = 0.0
        epoch_acc = 0.0
        n_batches = 0

        for i in range(0, n_samples, batch_size):
            X_batch = X_shuffled[i:i+batch_size]
            y_batch = y_shuffled[i:i+batch_size]

            z1, h1, z2, y_pred = forward_with_dropout(X_batch, w1, b1, w2, b2, dropout, is_training=True)

            loss = cross_entropy_loss(y_pred, y_batch)
            epoch_loss += loss
            epoch_acc += accuracy(y_pred, y_batch)
            n_batches += 1

            n = X_batch.shape[0]
            dz2 = y_pred.copy()
            dz2[np.arange(n), y_batch] -= 1
            dz2 /= n

            dw2 = h1.T @ dz2
            db2 = np.sum(dz2, axis=0)

            dh1 = dz2 @ w2.T
            dz1 = dh1 * (z1 > 0)

            dw1 = X_batch.T @ dz1
            db1 = np.sum(dz1, axis=0)

            w1 -= lr * dw1
            b1 -= lr * db1
            w2 -= lr * dw2
            b2 -= lr * db2

        avg_loss = epoch_loss / n_batches
        avg_acc = epoch_acc / n_batches
        loss_history.append(avg_loss)
        acc_history.append(avg_acc)

        if (epoch + 1) % 5 == 0:
            print(f"Epoch {epoch+1}/{epochs}, Loss: {avg_loss:.4f}, Acc: {avg_acc:.4f}")

    return w1, b1, w2, b2, loss_history, acc_history

In [ ]:
# 对比实验：使用高维多项式拟合

# 生成高维多项式数据
np.random.seed(42)

# 生成训练数据（少量样本）
n_train = 20
n_test = 100
degree = 15  # 高维多项式

X_train_poly = np.linspace(-3, 3, n_train).reshape(-1, 1)
y_train_poly = np.sin(X_train_poly) + np.random.normal(0, 0.1, X_train_poly.shape)

X_test_poly = np.linspace(-4, 4, n_test).reshape(-1, 1)
y_test_poly = np.sin(X_test_poly)

# 多项式特征扩展
def polynomial_features(X, degree):
    features = []
    for d in range(1, degree + 1):
        features.append(X ** d)
    return np.hstack(features)

X_train_features = polynomial_features(X_train_poly, degree)
X_test_features = polynomial_features(X_test_poly, degree)

print(f"特征维度: {X_train_features.shape[1]}")

In [ ]:
# 使用复杂MLP进行对比实验

# 实验参数
input_dim = degree
hidden_dim = 256
output_dim = 1
epochs = 100
lr = 0.01

# 训练函数（回归任务）
def train_regression_mlp(X_train, y_train, X_val, y_val, dropout=0.0, weight_decay=0.0):
    np.random.seed(42)
    w1 = np.random.randn(input_dim, hidden_dim) * 0.01
    b1 = np.zeros(hidden_dim)
    w2 = np.random.randn(hidden_dim, output_dim) * 0.01
    b2 = np.zeros(output_dim)
    
    train_loss_history = []
    val_loss_history = []
    
    for epoch in range(epochs):
        indices = np.random.permutation(len(X_train))
        X_shuffled = X_train[indices]
        y_shuffled = y_train[indices]

        # 训练
        z1 = X_shuffled @ w1 + b1
        h1 = relu(z1)
        if dropout > 0:
            h1 = dropout_layer(h1, dropout, is_training=True)
        y_pred = h1 @ w2 + b2
        
        # MSE损失
        train_loss = np.mean((y_pred - y_shuffled) ** 2)
        
        # 反向传播
        n = len(X_shuffled)
        dy = (y_pred - y_shuffled) / n
        dw2 = h1.T @ dy
        db2 = np.sum(dy, axis=0)
        dh1 = dy @ w2.T
        dz1 = dh1 * (z1 > 0)
        dw1 = X_shuffled.T @ dz1
        db1 = np.sum(dz1, axis=0)
        
        # 更新
        w1 = (1 - lr * weight_decay) * w1 - lr * dw1
        b1 -= lr * db1
        w2 = (1 - lr * weight_decay) * w2 - lr * dw2
        b2 -= lr * db2
        
        # 验证
        z1_val = X_val @ w1 + b1
        h1_val = relu(z1_val)
        if dropout > 0:
            h1_val = dropout_layer(h1_val, dropout, is_training=False)
        y_pred_val = h1_val @ w2 + b2
        val_loss = np.mean((y_pred_val - y_val) ** 2)
        
        train_loss_history.append(train_loss)
        val_loss_history.append(val_loss)
        
        if (epoch + 1) % 20 == 0:
            print(f"Epoch {epoch+1}, Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")
    
    return train_loss_history, val_loss_history

# 三种情况训练
print("=== 无正则化 ===")
train_loss_no_reg, val_loss_no_reg = train_regression_mlp(
    X_train_features, y_train_poly, X_test_features, y_test_poly,
    dropout=0.0, weight_decay=0.0
)

print("\n=== 权重衰减 ===")
train_loss_wd, val_loss_wd = train_regression_mlp(
    X_train_features, y_train_poly, X_test_features, y_test_poly,
    dropout=0.0, weight_decay=0.01
)

print("\n=== Dropout ===")
train_loss_dropout, val_loss_dropout = train_regression_mlp(
    X_train_features, y_train_poly, X_test_features, y_test_poly,
    dropout=0.5, weight_decay=0.0
)

In [ ]:
# 绘制对比曲线
plt.figure(figsize=(12, 6))

# 训练损失
plt.subplot(1, 2, 1)
plt.plot(train_loss_no_reg, label='无正则化', linestyle='-')
plt.plot(train_loss_wd, label='权重衰减', linestyle='--')
plt.plot(train_loss_dropout, label='Dropout', linestyle='-.')
plt.xlabel('Epoch')
plt.ylabel('训练损失')
plt.title('训练损失对比')
plt.legend()

# 验证损失
plt.subplot(1, 2, 2)
plt.plot(val_loss_no_reg, label='无正则化', linestyle='-')
plt.plot(val_loss_wd, label='权重衰减', linestyle='--')
plt.plot(val_loss_dropout, label='Dropout', linestyle='-.')
plt.xlabel('Epoch')
plt.ylabel('验证损失')
plt.title('验证损失对比')
plt.legend()

plt.tight_layout()
plt.show()

## 4 数值稳定性和激活函数

### 4.1 理论计算题

#### 1. 梯度消失与梯度爆炸

考虑一个 $d$ 层的深层神经网络，其梯度计算包含多层矩阵连乘项 $\prod_{i=t}^{d-1} \frac{\partial h^{i+1}}{\partial h^{i}}$。

**梯度爆炸的情况：**

当权重矩阵的谱范数（最大奇异值）大于1时，多层矩阵连乘会导致梯度范数指数增长。

假设每层的梯度缩放因子为 $\lambda_i = \|\frac{\partial h^{i+1}}{\partial h^{i}}\|_2$，则经过 $k$ 层后，梯度范数会放大为 $\prod_{i=t}^{t+k-1} \lambda_i$。

如果 $\lambda_i > 1$ 且 $k$ 较大，梯度范数会指数增长，导致梯度爆炸。

**梯度消失的情况：**

当使用Sigmoid或tanh激活函数时，其导数最大值分别为0.25和1。当输入绝对值较大时，导数趋近于0。

对于Sigmoid函数：$\sigma'(x) = \sigma(x)(1-\sigma(x)) \leq 0.25$

如果每层的梯度缩放因子 $\lambda_i < 1$，经过多层连乘后，梯度范数会指数衰减，导致梯度消失。

**量化分析：**

假设每层权重矩阵的谱范数为 $\sigma$，激活函数导数的上界为 $\gamma$，则第 $t$ 层的梯度范数满足：

\[
\|\frac{\partial L}{\partial h^t}\| \leq \|\frac{\partial L}{\partial h^d}\| \cdot (\sigma \cdot \gamma)^{d-t}
\]

- 当 $\sigma \cdot \gamma > 1$：梯度爆炸
- 当 $\sigma \cdot \gamma < 1$：梯度消失

#### 2. ReLU缓解梯度消失的原因

ReLU激活函数的数学表达式：

\[
\text{ReLU}(x) = \max(0, x)
\]

其导数：

\[
\text{ReLU}'(x) = \begin{cases}
1 & x > 0 \\
0 & x < 0
\end{cases}
\]

**ReLU缓解梯度消失的原因：**

1. **导数为常数1**：当输入大于0时，ReLU的导数恒为1，不会像Sigmoid那样趋近于0。这使得梯度可以有效地向后传播。

2. **无饱和区**：ReLU没有像Sigmoid和tanh那样的饱和区域。当输入很大时，导数仍然保持为1，不会衰减。

3. **稀疏激活**：一部分神经元输出为0（死亡ReLU），但存活的神经元梯度可以正常传播。

**注意：** ReLU仍可能出现"死亡ReLU"问题（当神经元长期处于非激活状态），可以使用LeakyReLU或Parametric ReLU来缓解。

### 4.2 编程题：模拟数值不稳定现象

In [ ]:
import torch
import torch.nn as nn

# 构建20层的深层全连接网络
class DeepNet(nn.Module):
    def __init__(self, hidden_size=256, activation='sigmoid', init_std=1.0):
        super(DeepNet, self).__init__()
        layers = []
        layers.append(nn.Linear(100, hidden_size))
        
        for _ in range(18):
            layers.append(nn.Linear(hidden_size, hidden_size))
        
        layers.append(nn.Linear(hidden_size, 10))
        
        self.layers = nn.ModuleList(layers)
        self.activation = activation
        
        # 初始化权重
        for m in self.layers:
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, mean=0, std=init_std)
                nn.init.zeros_(m.bias)
    
    def forward(self, x):
        for layer in self.layers[:-1]:
            x = layer(x)
            if self.activation == 'sigmoid':
                x = torch.sigmoid(x)
            elif self.activation == 'relu':
                x = torch.relu(x)
        x = self.layers[-1](x)
        return x

# 打印各层梯度范数
def print_grad_norms(net, prefix=''):
    print(f"{prefix}梯度范数:")
    for i, layer in enumerate(net.layers):
        if isinstance(layer, nn.Linear):
            grad_norm = layer.weight.grad.norm().item() if layer.weight.grad is not None else 0
            print(f"  层{i+1}: {grad_norm:.6e}")
            if torch.isnan(layer.weight.grad).any():
                print(f"  层{i+1}包含NaN!")

In [ ]:
# 实验1: Sigmoid + 普通高斯初始化 (std=1) - 梯度消失
print("=== 实验1: Sigmoid + 普通高斯初始化 (std=1) ===")
net1 = DeepNet(hidden_size=256, activation='sigmoid', init_std=1.0)
net1.train()

# 随机输入
X = torch.randn(32, 100)
y = torch.randint(0, 10, (32,))

# 前向传播
output = net1(X)
loss = nn.CrossEntropyLoss()(output, y)

# 反向传播
net1.zero_grad()
loss.backward()

print_grad_norms(net1)

In [ ]:
# 实验2: ReLU + 较大的初值 (std=10) - 梯度爆炸
print("\n=== 实验2: ReLU + 较大的初值 (std=10) ===")
net2 = DeepNet(hidden_size=256, activation='relu', init_std=10.0)
net2.train()

X = torch.randn(32, 100)
y = torch.randint(0, 10, (32,))

output = net2(X)
print(f"输出是否包含NaN: {torch.isnan(output).any()}")

if not torch.isnan(output).any():
    loss = nn.CrossEntropyLoss()(output, y)
    net2.zero_grad()
    loss.backward()
    print_grad_norms(net2)

In [ ]:
# 实验3: ReLU + Xavier初始化 - 梯度稳定
class DeepNetXavier(nn.Module):
    def __init__(self, hidden_size=256, activation='relu'):
        super(DeepNetXavier, self).__init__()
        layers = []
        layers.append(nn.Linear(100, hidden_size))
        
        for _ in range(18):
            layers.append(nn.Linear(hidden_size, hidden_size))
        
        layers.append(nn.Linear(hidden_size, 10))
        
        self.layers = nn.ModuleList(layers)
        self.activation = activation
        
        # Xavier初始化
        for m in self.layers:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)
    
    def forward(self, x):
        for layer in self.layers[:-1]:
            x = layer(x)
            if self.activation == 'relu':
                x = torch.relu(x)
            elif self.activation == 'leaky_relu':
                x = nn.LeakyReLU()(x)
        x = self.layers[-1](x)
        return x

print("\n=== 实验3: ReLU + Xavier初始化 ===")
net3 = DeepNetXavier(hidden_size=256, activation='relu')
net3.train()

X = torch.randn(32, 100)
y = torch.randint(0, 10, (32,))

output = net3(X)
loss = nn.CrossEntropyLoss()(output, y)
net3.zero_grad()
loss.backward()

print_grad_norms(net3)

## 5 泛化表现，协变量偏移和对抗性数据

### 5.1 理论计算题

#### 协变量偏移与标签偏移的区别与联系

**协变量偏移（Covariate Shift）**：

表现为 $p(x) \neq q(x)$ 但 $p(y | x) = q(y | x)$

即输入特征的分布发生了变化，但条件概率（标签与特征的关系）保持不变。

**例子（医疗诊断）**：

- 训练数据来自城市医院，患者年龄分布偏年轻
- 测试数据来自乡村医院，患者年龄分布偏老年
- 但"症状→疾病"的关系在两地是相同的

**标签偏移（Label Shift）**：

表现为 $p(y) \neq q(y)$ 但 $p(x | y) = q(x | y)$

即标签的边缘分布发生了变化，但给定标签下特征的条件分布保持不变。

**例子（电商推荐）**：

- 训练数据中"点击"和"不点击"的比例为1:1
- 测试数据中由于推广活动，"点击"比例变为3:1
- 但"用户特征→点击/不点击"的关系保持不变

**区别：**

| 特征 | 协变量偏移 | 标签偏移 |
|------|-----------|----------|
| 变化的分布 | $p(x)$ | $p(y)$ |
| 不变的分布 | $p(y|x)$ | $p(x|y)$ |
| 本质 | 输入分布漂移 | 类别比例漂移 |

**联系：**

1. 都是分布偏移的特殊情况
2. 都可能导致模型泛化性能下降
3. 都可以通过权重修正等方法缓解
4. 当特征和标签独立时，两者等价

### 5.2 编程题：协变量偏移模拟与权重修正

In [ ]:
# 构造人工数据集
np.random.seed(42)

# 训练集P：N(-1, 1)
n_train = 1000
X_train_shift = np.random.normal(-1, 1, n_train).reshape(-1, 1)
epsilon = np.random.normal(0, 0.1, n_train).reshape(-1, 1)
y_train_shift = 2 * X_train_shift + epsilon

# 测试集Q：N(2, 1)
n_test = 500
X_test_shift = np.random.normal(2, 1, n_test).reshape(-1, 1)
epsilon_test = np.random.normal(0, 0.1, n_test).reshape(-1, 1)
y_test_shift = 2 * X_test_shift + epsilon_test

print(f"训练集X均值: {X_train_shift.mean():.2f}, 标准差: {X_train_shift.std():.2f}")
print(f"测试集X均值: {X_test_shift.mean():.2f}, 标准差: {X_test_shift.std():.2f}")

# 可视化数据分布
plt.figure(figsize=(10, 5))
plt.hist(X_train_shift, bins=30, alpha=0.5, label='训练集P', density=True)
plt.hist(X_test_shift, bins=30, alpha=0.5, label='测试集Q', density=True)
plt.xlabel('x')
plt.ylabel('密度')
plt.title('训练集与测试集的特征分布')
plt.legend()
plt.show()

In [ ]:
# 基线模型：普通线性回归
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

# 训练线性回归模型
baseline_model = LinearRegression()
baseline_model.fit(X_train_shift, y_train_shift)

# 在测试集上评估
y_pred_baseline = baseline_model.predict(X_test_shift)
mse_baseline = mean_squared_error(y_test_shift, y_pred_baseline)

print(f"基线模型参数: w={baseline_model.coef_[0][0]:.4f}, b={baseline_model.intercept_[0]:.4f}")
print(f"基线模型测试MSE: {mse_baseline:.4f}")

# 可视化拟合结果
plt.figure(figsize=(10, 5))
plt.scatter(X_train_shift, y_train_shift, c='blue', label='训练集', alpha=0.5)
plt.scatter(X_test_shift, y_test_shift, c='red', label='测试集', alpha=0.5)
plt.plot(X_train_shift, baseline_model.predict(X_train_shift), 'k-', label='基线模型')
plt.xlabel('x')
plt.ylabel('y')
plt.title('基线模型拟合结果')
plt.legend()
plt.show()

In [ ]:
# 偏移校正：训练逻辑回归分类器
from sklearn.linear_model import LogisticRegression

# 构造分类数据
X_combined = np.vstack([X_train_shift, X_test_shift])
y_combined = np.array([0] * n_train + [1] * n_test)  # 0: 训练集, 1: 测试集

# 训练逻辑回归
domain_classifier = LogisticRegression()
domain_classifier.fit(X_combined, y_combined)

# 预测每个样本属于测试集的概率
p_test_given_x = domain_classifier.predict_proba(X_train_shift)[:, 1]
p_train_given_x = 1 - p_test_given_x

# 计算权重 w_i ∝ P(test|x_i) / P(train|x_i)
weights = p_test_given_x / p_train_given_x
weights = weights / weights.mean()  # 归一化

print(f"权重统计 - 均值: {weights.mean():.4f}, 标准差: {weights.std():.4f}")
print(f"权重范围: [{weights.min():.4f}, {weights.max():.4f}]")

# 可视化权重分布
plt.figure(figsize=(10, 5))
plt.scatter(X_train_shift, weights, c='green', alpha=0.6)
plt.xlabel('x')
plt.ylabel('权重')
plt.title('训练样本的权重分布')
plt.show()

In [ ]:
# 加权模型训练：加权最小二乘法

# 使用加权最小二乘法
# 权重较大的样本会有更大的影响力
weighted_model = LinearRegression()
weighted_model.fit(X_train_shift, y_train_shift, sample_weight=weights)

# 在测试集上评估
y_pred_weighted = weighted_model.predict(X_test_shift)
mse_weighted = mean_squared_error(y_test_shift, y_pred_weighted)

print(f"加权模型参数: w={weighted_model.coef_[0][0]:.4f}, b={weighted_model.intercept_[0]:.4f}")
print(f"加权模型测试MSE: {mse_weighted:.4f}")
print(f"\n校正前后MSE对比:")
print(f"  校正前: {mse_baseline:.4f}")
print(f"  校正后: {mse_weighted:.4f}")
print(f"  改善比例: {(mse_baseline - mse_weighted) / mse_baseline * 100:.2f}%")

# 可视化对比
plt.figure(figsize=(10, 5))
plt.scatter(X_train_shift, y_train_shift, c='blue', label='训练集', alpha=0.5)
plt.scatter(X_test_shift, y_test_shift, c='red', label='测试集', alpha=0.5)
plt.plot(X_train_shift, baseline_model.predict(X_train_shift), 'k-', label='基线模型')
plt.plot(X_train_shift, weighted_model.predict(X_train_shift), 'g--', label='加权模型')
plt.xlabel('x')
plt.ylabel('y')
plt.title('基线模型 vs 加权模型')
plt.legend()
plt.show()

# 总结

本作业完成了以下内容：

## 2 多层感知机
- 理论证明了无激活函数的MLP等价于单层线性模型
- 推导了Sigmoid和tanh的导数与自身的关系
- 从零实现了单隐藏层MLP，包含ReLU、Softmax和交叉熵损失

## 3 模型选择，权重衰减和丢弃法
- 解释了过拟合与欠拟合的区别及缓解方法
- 阐述了K折交叉验证的步骤
- 实现了带权重衰减的SGD和Dropout机制
- 通过高维多项式拟合对比了三种正则化方法的效果

## 4 数值稳定性和激活函数
- 量化分析了梯度消失和梯度爆炸的条件
- 解释了ReLU缓解梯度消失的原因
- 模拟了不同初始化策略对深层网络的影响

## 5 泛化表现，协变量偏移和对抗性数据
- 结合实例阐述了协变量偏移和标签偏移的区别与联系
- 构造了协变量偏移数据集
- 通过权重修正改善了模型在偏移测试集上的性能